# **Homework 3**
### *Machine Learning for Classification*

In this dataset our desired target for classification task will be `converted` variable - has the client signed up to the platform or not.

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(data)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


## **Data Prep**

In [4]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [5]:
target = "converted"

feature_cols = [c for c in df.columns if c != target]
numerical = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
categorical = [c for c in feature_cols if c not in numerical]

print("Categorical:", categorical)
print("Numerical:  ", numerical)

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum().sum()

Categorical: ['lead_source', 'industry', 'employment_status', 'location']
Numerical:   ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


np.int64(0)

## **Question 1 - Mode for industry**

In [8]:
print(df["industry"].value_counts())

print("Mode:", df["industry"].mode()[0])

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64
Mode: technology


## **Question 2 - Biggest correlation**

In [9]:
corr = df[numerical].corr()
corr.round(3)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


In [18]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
for a, b in pairs:
    print(f"{a:26s} & {b:18s}: {corr.loc[a, b]:.4f}")

interaction_count          & lead_score        : 0.9157
number_of_courses_viewed   & lead_score        : 0.7572
number_of_courses_viewed   & interaction_count : 0.7216
annual_income              & interaction_count : 0.1228


In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## **Question 3 - Biggest MI**

In [12]:
mi = {c: round(mutual_info_score(y_train, df_train[c]), 2) for c in categorical}
pd.Series(mi).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## **Question 4 - Accuracy**

In [17]:
features = categorical + numerical

def train_and_score(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[feats].to_dict(orient="records"))
    X_val = dv.transform(df_val[feats].to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

base_acc = train_and_score(features)
print(round(base_acc, 2))

0.65


## **Question 5 - Feature Selection**

In [14]:
diffs = {}
for f in features:
    feats = [x for x in features if x != f]
    acc = train_and_score(feats)
    diffs[f] = base_acc - acc

pd.Series(diffs).sort_values()

annual_income              -0.079
industry                    0.000
location                    0.000
employment_status           0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

In [16]:
candidates = ["lead_source", "number_of_courses_viewed", "interaction_count"]
for f in candidates:
    print(f"{f:26s}: {diffs[f]:+.5f}")

lead_source               : +0.00300
number_of_courses_viewed  : +0.00200
interaction_count         : +0.04400


## **Question 6 - Parameter Tuning**

In [22]:
results = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    results[C] = round(train_and_score(features, C=C), 3)
    print(f"C={C:<10}: {results[C]}")

C=1e-06     : 0.598
C=1e-05     : 0.598
C=0.0001    : 0.613
C=0.001     : 0.645
